# Dictionaries

**Topic:** `3.4` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Fetch required and optional config keys correctly

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** KeyError, get defaults, contract design

**Explanation and reasoning**

The two access styles encode two contracts on adjacent lines: bracket access fails AT THE SOURCE with the missing key named, while get() treats absence as a documented default. A try/except would work but would also swallow real KeyErrors raised by mistake inside the block.

In [ ]:
def read_config(config):
    """host is required (KeyError if absent); timeout defaults to 30."""
    host = config['host']                  # contract: must exist
    timeout = int(config.get('timeout', 30))
    return host, timeout

**Complexity**

Time O(1) average per access; space O(1).

**Edge cases and failure modes**

Missing host raises KeyError('host'). Missing timeout returns 30. String timeouts coerce.

**Alternative approaches**

config.pop('timeout', 30) reads the default too but destroys the caller's dict.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert read_config({'host': 'mars.local', 'timeout': 5}) == ('mars.local', 5)
assert read_config({'host': 'x'}) == ('x', 30)
try:
    read_config({'timeout': 9})
    raise AssertionError('should raise')
except KeyError as exc:
    assert 'host' in str(exc)

---

## Solution 2 — Count frequencies with a dict

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** counting idiom, get, max with key

**Explanation and reasoning**

get(item, 0) + 1 is the counting idiom: the default only exists to seed the first sighting, and insertion order then guarantees max() walks candidates in first-seen order - so ties resolve to the earliest key deterministically because max keeps the first maximum it meets.

In [ ]:
def frequencies(items):
    """item -> occurrence count, one pass."""
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1
    return counts


def most_common(counts):
    """Key with the highest count; ties go to first insertion."""
    if not counts:
        raise ValueError('no counts to inspect')
    return max(counts, key=counts.get)

**Complexity**

Time O(n) expected for counting, O(k) for the max; space O(k) for k distinct keys.

**Edge cases and failure modes**

Empty input counts to {} and most_common raises ValueError. All-unique input gives first key.

**Alternative approaches**

collections.Counter(items).most_common(1) does both but hides the mechanism being taught.

**Tests — run the cell to verify the reference solution**

In [ ]:
counts = frequencies(['a', 'b', 'a'])
assert counts == {'a': 2, 'b': 1}
assert most_common(counts) == 'a'
assert most_common({'x': 1, 'y': 1}) == 'x'
try:
    most_common({})
    raise AssertionError('should raise')
except ValueError:
    pass

---

## Solution 3 — Merge parameter sets with override semantics

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** merge operator, update, immutability of inputs

**Explanation and reasoning**

The {**d} unpacking form builds a fresh dict from each operand's items, so precedence falls out of evaluation order (the later unpack overwrites) and neither input is touched. `base | override` is equivalent and equally non-mutating - both beat update(), which would edit base in the caller's hands.

In [ ]:
def merge_params(base, override):
    """New dict with override winning; inputs untouched."""
    return {**base, **override}

**Complexity**

Time O(len(base) + len(override)); space O(same) for the result.

**Edge cases and failure modes**

Empty override returns an equal new dict. Empty base returns a copy of override.

**Alternative approaches**

dict(base); merged.update(override) is two statements saying the same thing.

**Tests — run the cell to verify the reference solution**

In [ ]:
base = {'port': 80}
over = {'port': 443, 'tls': True}
out = merge_params(base, over)
assert out == {'port': 443, 'tls': True}
assert base == {'port': 80} and over == {'port': 443, 'tls': True}

---

## Solution 4 — Group events into a dict of lists

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** grouping, setdefault, append

**Explanation and reasoning**

setdefault(kind, []) returns the existing list when the key is present and installs-and-returns it on the first sighting, so one line covers both branches. Dict insertion order then gives first-encounter key order, and list order gives detail order - both required by the specification.

In [ ]:
def group(events):
    """kind -> list of details, in encounter order."""
    groups = {}
    for kind, detail in events:
        groups.setdefault(kind, []).append(detail)
    return groups

**Complexity**

Time O(total details) expected; space O(total details).

**Edge cases and failure modes**

Empty input gives {}. A single kind collects every detail.

**Alternative approaches**

The explicit get-then-create avoids building a throwaway list on hits (see lesson advanced examples).

**Tests — run the cell to verify the reference solution**

In [ ]:
out = group([('imu', 'boot'), ('gps', 'fix'), ('imu', 'drift')])
assert out == {'imu': ['boot', 'drift'], 'gps': ['fix']}
assert group([]) == {}

---

## Solution 5 — Invert a mapping with an explicit collision policy

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** dict comprehension, collision policy, reverse index

**Explanation and reasoning**

Insertion order makes both policies mechanical: 'first' skips values already present (the first key got there first), 'last' overwrites unconditionally (the last writer wins). Validating the policy up front turns a silent wrong answer on a typo into a loud failure.

In [ ]:
def invert(mapping, policy='last'):
    """value -> key with an explicit collision policy."""
    if policy not in ('first', 'last'):
        raise ValueError(f'unknown policy {policy!r}')
    reversed_map = {}
    for key, value in mapping.items():
        if policy == 'first' and value in reversed_map:
            continue                 # keep the earliest key
        reversed_map[value] = key      # 'last' simply overwrites
    return reversed_map

**Complexity**

Time O(n) expected; space O(n).

**Edge cases and failure modes**

Empty dict inverts to {}. Invalid policy raises ValueError before any work.

**Alternative approaches**

collections.defaultdict(list) of ALL keys per value trades the policy for more structure.

**Tests — run the cell to verify the reference solution**

In [ ]:
m = {'a': 1, 'b': 1}
assert invert(m, 'first') == {1: 'a'}
assert invert(m, 'last') == {1: 'b'}
assert invert({}) == {}
try:
    invert(m, 'middle')
    raise AssertionError('should raise')
except ValueError:
    pass

---

## Solution 6 — Deep-copy a nested record without sharing state

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** shallow vs deep copy, recursion, isolation

**Explanation and reasoning**

Each container type is rebuilt rather than referenced, so no result node is `is` a source node - the property the exercise's assertions check with id(). Scalars are returned as-is because sharing immutable objects is both safe and free; cloning them would cost memory for nothing.

In [ ]:
def deep_copy(record):
    """Recursively clone dicts/lists; scalars pass through."""
    if isinstance(record, dict):
        return {k: deep_copy(v) for k, v in record.items()}
    if isinstance(record, list):
        return [deep_copy(v) for v in record]
    if isinstance(record, tuple):
        return tuple(deep_copy(v) for v in record)
    return record   # scalars are immutable - sharing is safe

**Complexity**

Time O(n) over all nodes; space O(n) for the clone.

**Edge cases and failure modes**

Empty dicts return new empties. Tuples are rebuilt so nested dicts inside them are still cloned.

**Alternative approaches**

copy.deepcopy is one line but is exactly the library call the exercise asks you to understand.

**Tests — run the cell to verify the reference solution**

In [ ]:
src = {'cfg': {'mode': 'a'}, 'tags': [1, 2]}
clone = deep_copy(src)
assert clone == src
assert clone['cfg'] is not src['cfg']
clone['cfg']['mode'] = 'b'
assert src['cfg']['mode'] == 'a'

---

## Solution 7 — Build an O(1) reverse lookup with collision handling

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** reverse index, precomputation, collision lists

**Explanation and reasoning**

All the work happens once in the constructor, so lookup is a single get - O(1) expected, never a scan. try/except around setdefault catches the TypeError that hashing an unhashable value raises, counting it instead of failing the build, and sorting buckets up front keeps lookups reproducible.

In [ ]:
class ReverseIndex:
    """Precomputed value -> keys index with unhashable tracking."""

    def __init__(self, mapping):
        self._index = {}
        self.unindexable = 0
        for key, value in mapping.items():
            try:
                bucket = self._index.setdefault(value, [])
            except TypeError:            # unhashable value
                self.unindexable += 1
                continue
            bucket.append(key)
        for value in self._index:
            self._index[value].sort()
        self.size = len(mapping)

    def lookup(self, value):
        """Sorted keys for a hashable value; [] when absent."""
        return list(self._index.get(value, ()))

**Complexity**

Build O(n log n) from bucket sorts; lookup O(1) expected.

**Edge cases and failure modes**

Empty mapping gives size 0. List values increment unindexable and never enter the index.

**Alternative approaches**

Skipping the try/except and pre-checking with hash(value) doubles the hashing work per entry.

**Tests — run the cell to verify the reference solution**

In [ ]:
idx = ReverseIndex({'a': 1, 'b': 1, 'c': [2]})
assert idx.lookup(1) == ['a', 'b']
assert idx.unindexable == 1 and idx.size == 3
assert idx.lookup(99) == []
assert ReverseIndex({}).size == 0

---

## Solution 8 — Detect and repair aliasing in a settings tree

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** identity testing, alias detection, repair

**Explanation and reasoning**

Both functions key their bookkeeping on id(): is_independent flags any container whose id it has already seen (a shared node, or a cycle), and detach memoises clones so a node reached twice is cloned once - the memo doubles as the cycle breaker, which is why clone can never recurse forever.

In [ ]:
def is_independent(tree):
    """True when no dict/list node is reachable twice."""
    visited = set()

    def walk(node):
        if not isinstance(node, (dict, list)):
            return True
        if id(node) in visited:
            return False
        visited.add(id(node))
        children = node.values() if isinstance(node, dict) else node
        return all(walk(child) for child in children)

    return walk(tree)


def detach(tree):
    """Clone the tree, memoising nodes so shared/cyclic paths stay consistent."""
    memo = {}

    def clone(node):
        if not isinstance(node, (dict, list)):
            return node
        if id(node) in memo:
            return memo[id(node)]
        if isinstance(node, dict):
            copy_node = {}
        else:
            copy_node = []
        memo[id(node)] = copy_node
        if isinstance(node, dict):
            for k, v in node.items():
                copy_node[k] = clone(v)
        else:
            copy_node.extend(clone(v) for v in node)
        return copy_node

    return clone(tree)

**Complexity**

Time O(n) per pass; space O(depth) plus the visited/memo sets.

**Edge cases and failure modes**

Scalar-only trees are independent. detach of an independent tree still returns a fresh clone. Cycles terminate.

**Alternative approaches**

copy.deepcopy handles cycles too, but hides the id()-memo mechanism this exercise is about.

**Tests — run the cell to verify the reference solution**

In [ ]:
shared = {}
assert is_independent({'a': shared, 'b': shared}) is False
fixed = detach({'a': shared, 'b': shared})
assert is_independent(fixed) is True
assert is_independent({'x': [1, {'y': 2}]}) is True

---

## Solution 9 — Implement a typed settings table with validation

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** schema-driven validation, type checks, error accumulation

**Explanation and reasoning**

Three passes' worth of checks run in two loops over already-known keys: schema-first gives missing/type in schema order (so `clean` inherits that order), values-second finds extras, and sorting at the end makes the error report reproducible regardless of dict order. The explicit bool guard prevents True from satisfying int because bool subclasses int.

In [ ]:
def validate_settings(values, schema):
    """(clean in schema order, sorted error messages) - never raises."""
    clean, errors = {}, []
    for key, expected in schema.items():
        if key not in values:
            errors.append(f'missing: {key}')
        elif isinstance(values[key], expected) and not (
            expected is int and isinstance(values[key], bool)
        ):
            clean[key] = values[key]
        else:
            errors.append(f'type: {key}')
    for key in values:
        if key not in schema:
            errors.append(f'unexpected: {key}')
    return clean, sorted(errors)

**Complexity**

Time O(s + v + e log e); space O(s + e).

**Edge cases and failure modes**

Exact match gives (copy, []). bool for int reports a type error. Extras report 'unexpected'.

**Alternative approaches**

Raising on the first error is simpler but forces repair loops - the exercise demands a full report.

**Tests — run the cell to verify the reference solution**

In [ ]:
clean, errors = validate_settings({'rate': 10}, {'rate': int, 'host': str})
assert clean == {'rate': 10}
assert errors == ['missing: host']
clean, errors = validate_settings({'rate': True, 'x': 1}, {'rate': int})
assert errors == ['type: rate', 'unexpected: x']

---

## Solution 10 — Simulate a parameter-server update cycle

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** snapshot isolation, staged updates, nested merge

**Explanation and reasoning**

The first loop deep-clones live so every container in the result is fresh - the isolation guarantee that lets the second loop mutate `result` freely without touching live at any depth. The merge recurses only when BOTH sides are dicts (otherwise updates' non-dict value replaces wholesale), and the None sentinel pops from the clone, so deletions never reach the caller's live tree.

In [ ]:
def apply_updates(live, updates):
    """Deep-merge updates into a NEW tree; None deletes; live untouched."""
    result = {}
    for key, value in live.items():
        if isinstance(value, dict):
            result[key] = apply_updates(value, {})
        elif isinstance(value, list):
            result[key] = list(value)
        else:
            result[key] = value
    for key, value in updates.items():
        if value is None:
            result.pop(key, None)          # delete from RESULT only
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = apply_updates(result[key], value)
        else:
            result[key] = value
    return result

**Complexity**

Time O(size of both trees); space O(size of live) for the clone.

**Edge cases and failure modes**

Empty updates returns a full clone. None on a missing key is a no-op. Type mismatches replace.

**Alternative approaches**

copy.deepcopy(live) then recursive update is shorter but imports the copy module the exercise forbids.

**Tests — run the cell to verify the reference solution**

In [ ]:
live = {'a': {'x': 1, 'y': 2}, 'b': 3}
out = apply_updates(live, {'a': {'y': 9, 'z': 3}, 'b': None})
assert out == {'a': {'x': 1, 'y': 9, 'z': 3}}
assert live == {'a': {'x': 1, 'y': 2}, 'b': 3}
clone = apply_updates(live, {})
clone['a']['x'] = 999
assert live['a']['x'] == 1

---

# Robotics challenge solution — ROBO-X Challenge: Parameter Server with Atomic Updates

**Explanation**

Reusing the deep-merge from exercise 10 makes the parameter server's update cycle non-destructive by construction: defaults are cloned, overrides win per key, None deletes from the staged tree, and validation runs on the staged result BEFORE anyone binds it as live - so a rejected update cannot half-apply.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402


def load_params(robot, overrides=None):
    """Merge defaults with overrides into an independent tree."""
    defaults = {
        'speed_mps': 0.4,
        'gains': {'kp': 1.2, 'ki': 0.05},
        'name': robot.name,
    }
    merged = apply_updates(defaults, overrides or {})
    if merged['speed_mps'] <= 0:
        raise ValueError('speed_mps must be positive')
    return merged

**Complexity**

Time O(params); space O(params) for the independent tree.

**Notes and trade-offs**

No overrides returns a full clone of defaults. A zero speed raises before returning.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
p = load_params(robot, {'gains': {'kp': 2.0}})
assert p['gains'] == {'kp': 2.0, 'ki': 0.05}
try:
    load_params(robot, {'speed_mps': 0})
    raise AssertionError('should raise')
except ValueError:
    pass

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.